In [1]:
import numpy as np
import pandas as pd

In [4]:
data = pd.read_csv('./data/data.tsv',sep='\t',header=None)

In [9]:
windows = 5
half_window = windows//2
head = ['kmer'] + ['mean_{}'.format(i - half_window) for i in range(windows)] \
       + ['std_{}'.format(i - half_window) for i in range(windows)] \
       + ['dwell_{}'.format(i - half_window) for i in range(windows)] + ['signal', 'label', 'read_id', 'position']
data.columns = head
data

,kmer,mean_-2,mean_-1,mean_0,mean_1,mean_2,std_-2,std_-1,std_0,std_1,std_2,dwell_-2,dwell_-1,dwell_0,dwell_1,dwell_2,signal,label,read_id,position
0,AAGAA,0.259136,-0.769860,-1.323515,-1.411942,-2.174555,0.626005,0.114877,0.201130,0.091860,0.189352,23,3,10,3,15,0.04682265964148272*-0.5952591638512094*-0.696...,1,0371ba95-c5f8-4181-b05e-3c0998f1ce71,55
1,AAGAA,-2.005093,0.300276,0.697353,0.969393,-0.262328,0.197409,0.109504,0.353996,0.177087,0.628766,16,3,6,5,27,-1.8963197009285064*-2.2511543928586786*-2.166...,1,0371ba95-c5f8-4181-b05e-3c0998f1ce71,239
2,AAGAA,0.095333,-0.855471,-1.158489,-2.008966,-1.819439,0.653931,0.151357,0.048451,0.166891,0.117367,31,15,6,3,20,-0.4938778232997317*-0.6290529440350352*-0.460...,1,0371ba95-c5f8-4181-b05e-3c0998f1ce71,423
3,AAGAA,-1.428839,-0.295429,-1.483553,-1.417574,-1.045843,0.131125,0.905907,0.118056,0.007965,0.243039,3,47,14,3,3,-1.4232067783549438*-1.5921756792740733*-1.271...,1,0371ba95-c5f8-4181-b05e-3c0998f1ce71,469
4,AAGAA,-1.836237,-1.880371,-1.504832,-1.426194,0.098755,0.199543,0.162060,0.125651,0.112059,0.594222,3,7,5,3,37,-1.8418541544807252*-2.0777695581395177*-1.589...,1,0bd9926c-3ed4-4006-ae9d-4462a4346ba0,55
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
334,AAGAA,-1.825268,0.251610,-1.358438,-1.323206,-0.765359,0.191203,0.771815,0.102046,0.183965,0.429665,6,16,3,12,3,-1.493495618824365*-1.757738769862316*-1.96913...,1,fdb5f88e-63b3-4f69-b1b9-34a7a3b49158,101
335,AAGAA,-1.271531,0.233929,-0.821878,-1.180808,-1.217508,0.132345,0.697971,0.186523,0.197496,0.072396,10,34,8,4,3,-1.4582631986859715*-1.1235552073712338*-1.211...,1,fdb5f88e-63b3-4f69-b1b9-34a7a3b49158,147
336,AAGAA,-1.845820,-0.198704,-1.444170,-1.238061,-0.976753,0.176748,0.566135,0.149229,0.176256,0.036198,3,22,20,14,3,-1.9339008705542833*-2.00436571083107*-1.59919...,1,fdb5f88e-63b3-4f69-b1b9-34a7a3b49158,239
337,AAGAA,-1.694691,-0.075941,-1.587449,-1.654978,-1.434775,0.164096,0.876451,0.081788,0.199931,0.221121,19,32,3,6,3,-1.563960459101152*-1.4582631986859715*-1.7225...,1,fdb5f88e-63b3-4f69-b1b9-34a7a3b49158,285


In [229]:
def parse_raw(info: str, l: int = 125):
    arr = np.fromstring(info, sep='*')
    
    if len(arr) < l:
        return np.pad(arr, (0, l - len(arr)), mode='constant', constant_values=0)
    else:
        return arr[:l]


def dataframe2npz(df: pd.DataFrame,windows = 5):
    half_window = windows//2
    signal = np.array(df['signal'].apply(parse_raw).tolist())
    label = df['label'].values
    read_id = np.array(df['read_id'].tolist())
    kmers = np.array(df['kmer'].tolist())
    mean = df[['mean_{}'.format(i - half_window) for i in range(windows)]].values
    std = df[['std_{}'.format(i - half_window) for i in range(windows)]].values
    dwell = df[['dwell_{}'.format(i - half_window) for i in range(windows)]].values
    basecall_pos = df[['position']].values
    return {
        'signal': signal,
        'label': label,
        'read_id': read_id,
        'kmers': kmers,
        'mean': mean,
        'std': std,
        'dwell': dwell,
        'basecall_pos': basecall_pos
    }

In [230]:
dic = dataframe2npz(data)
signal = dic['signal']

In [233]:
dic.keys()

dict_keys(['signal', 'label', 'read_id', 'kmers', 'mean', 'std', 'dwell', 'basecall_pos'])

In [234]:
np.savez_compressed('./data/dic.npz', **dic)

In [235]:
dic2  = np.load('./data/dic.npz')
dic2.files

['signal', 'label', 'read_id', 'kmers', 'mean', 'std', 'dwell', 'basecall_pos']

In [236]:
dic2['signal']

array([[ 0.04682266, -0.59525916, -0.6966405 , ...,  0.        ,
         0.        ,  0.        ],
       [-1.8963197 , -2.25115439, -2.16666994, ...,  0.        ,
         0.        ,  0.        ],
       [-0.49387782, -0.62905294, -0.46008404, ...,  0.        ,
         0.        ,  0.        ],
       ...,
       [-1.93390087, -2.00436571, -1.59919288, ...,  0.        ,
         0.        ,  0.        ],
       [-1.56396046, -1.4582632 , -1.72250635, ...,  0.        ,
         0.        ,  0.        ],
       [-1.40541457, -1.58157667, -1.22925247, ...,  0.        ,
         0.        ,  0.        ]])

# esox

In [239]:
data = pd.read_csv('./data/data_esox.tsv',sep='\t',header=None)

In [261]:
h = ['kmer' , 'read_id' , 'position', 'x1' ,'e1' , 's1' , 'label']
data.columns = h
data

,kmer,read_id,position,x1,e1,s1,label
0,AAGAA,01bd9cb6-541a-42f0-a040-69fa6070b674,9,1.0207006430737855*0.8881882286507751*0.871624...,1.1502224206924438*1.1502224206924438*1.150222...,_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*T*_*_*G*_*_*...,1
1,AAGAA,01bd9cb6-541a-42f0-a040-69fa6070b674,101,1.1863411611025487*1.4679300417514458*1.534186...,1.4628180265426636*1.4628180265426636*1.462818...,_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*...,1
2,AAGAA,01bd9cb6-541a-42f0-a040-69fa6070b674,147,-1.4639071273576607*-1.7951881634151867*-1.679...,-0.7762674689292908*-0.7762674689292908*-0.776...,_*_*_*_*_*_*_*_*C*_*_*_*_*_*_*_*_*_*_*_*_*A*_*...,1
3,AAGAA,01bd9cb6-541a-42f0-a040-69fa6070b674,193,-1.1160620394972582*-1.1160620394972582*-0.470...,-0.7231087684631348*-0.7231087684631348*-0.723...,_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*C*...,1
4,AAGAA,0371ba95-c5f8-4181-b05e-3c0998f1ce71,9,1.1726020337303182*1.002979288675181*0.8503188...,1.0610090494155884*1.0610090494155884*1.061009...,_*_*_*_*_*_*_*_*_*_*_*T*_*_*G*_*_*_*_*_*_*_*_*...,1
...,...,...,...,...,...,...,...
733,AAGAA,fdb5f88e-63b3-4f69-b1b9-34a7a3b49158,101,-1.1235552073712338*-1.0178579469560534*-1.246...,-0.9999296069145203*-0.9999296069145203*-0.999...,_*_*_*_*_*_*_*_*_*_*_*T*_*_*_*_*_*_*_*_*G*_*_*...,1
734,AAGAA,fdb5f88e-63b3-4f69-b1b9-34a7a3b49158,147,-1.4758794087551683*-2.00436571083107*-1.42303...,-1.8202754259109497*-1.8202754259109497*-1.820...,_*_*_*_*_*_*A*_*_*_*_*_*_*_*_*_*A*_*_*_*_*_*_*...,1
735,AAGAA,fdb5f88e-63b3-4f69-b1b9-34a7a3b49158,239,0.5323685391332587*0.9903900009323735*1.043238...,1.0610090494155884*1.0610090494155884*1.061009...,_*_*_*_*_*_*_*_*T*_*_*G*_*_*G*_*_*G*_*_*_*_*_*...,1
736,AAGAA,fdb5f88e-63b3-4f69-b1b9-34a7a3b49158,285,-1.4582631986859715*-1.7225063497239226*-1.511...,-1.9457216262817383*-1.9457216262817383*-1.945...,_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*A*_*_*_*_*...,1


In [269]:
def parse_esox(info: str):
    arr = np.fromstring(info, sep='*')
    return arr

ENCODING_DICT_CRF = {
    "_" : 0,
    "A": 1,
    "C": 2,
    "G": 3,
    "T": 4,
    "o": 5,
    "a": 1,
    "c": 2,
    "g": 3,
    "t": 4,
}
def parse_seq(seq:str):
    arr = np.array(seq.split('*'))
    return np.vectorize(ENCODING_DICT_CRF.get)(arr)


def dataframe2npz_esox(data: pd.DataFrame):
    
    return {
        'x1' : np.array(data['x1'].apply(parse_esox).tolist()),
        'e1' : np.array(data['e1'].apply(parse_esox).tolist()),
        's1' : np.array(data['s1'].apply(parse_seq).tolist()),
        'read_id' : np.array(data['read_id'].tolist()),
        'label' : data['label'].values,
        'basecall_pos' : data[['position']].values,
        'kmers' : np.array(data['kmer'].tolist())
    }

In [270]:
dic_esox = dataframe2npz_esox(data)

In [275]:
np.savez_compressed('./data/dic_esox.npz',**dic_esox)

In [276]:
dic_espx2 = np.load('./data/dic_esox.npz')

In [278]:
dic_espx2.files

['x1', 'e1', 's1', 'read_id', 'label', 'basecall_pos', 'kmers']

In [281]:
dic_espx2['x1']

array([[ 1.02070064,  0.88818823,  0.87162418, ...,  0.47408693,
         0.2090621 ,  1.03726469],
       [ 1.18634116,  1.46793004,  1.53418625, ...,  2.61084962,
         2.16362022,  0.54034314],
       [-1.46390713, -1.79518816, -1.6792398 , ...,  1.56731435,
        -1.36452282, -1.36452282],
       ...,
       [ 0.53236854,  0.99039   ,  1.04323863, ...,  2.76962722,
         0.49713612,  0.21527676],
       [-1.4582632 , -1.72250635, -1.51111183, ...,  1.06085484,
         1.06085484,  0.6380658 ],
       [-1.54634425, -1.65204151, -1.9867495 , ...,  1.37794662,
         1.3427142 ,  1.25463315]])

In [282]:
info = np.load('./data/infotrain.npz')

In [283]:
info.files

['signal', 'label', 'read_id', 'kmers', 'mean', 'std', 'dwell', 'basecall_pos']

In [292]:
info['basecall_pos']

array([[ 101],
       [ 469],
       [ 147],
       [ 147],
       [ 837],
       [ 101],
       [ 147],
       [ 285],
       [ 331],
       [ 561],
       [ 239],
       [ 285],
       [ 975],
       [ 377],
       [   9],
       [ 193],
       [  55],
       [ 883],
       [ 147],
       [ 101],
       [ 561],
       [ 193],
       [ 101],
       [ 285],
       [   9],
       [ 653],
       [ 515],
       [ 377],
       [ 423],
       [ 101],
       [ 239],
       [ 147],
       [ 285],
       [ 101],
       [ 193],
       [ 377],
       [ 147],
       [ 331],
       [ 193],
       [ 193],
       [ 331],
       [ 101],
       [   9],
       [ 101],
       [ 469],
       [  55],
       [ 653],
       [ 377],
       [  55],
       [ 469],
       [ 607],
       [ 331],
       [ 561],
       [ 101],
       [ 377],
       [ 193],
       [  55],
       [ 239],
       [ 193],
       [  55],
       [ 607],
       [ 469],
       [ 101],
       [   9],
       [ 193],
       [ 193],
       [  

In [293]:
info = np.load('./data/info_esox_train.npz')

In [294]:
info.files

['x1', 'e1', 's1', 'read_id', 'label', 'basecall_pos', 'kmers']

In [299]:
info['basecall_pos']

array([[   9],
       [  55],
       [ 193],
       [ 101],
       [ 883],
       [ 147],
       [ 239],
       [ 193],
       [ 607],
       [  55],
       [ 239],
       [ 607],
       [ 561],
       [ 331],
       [ 101],
       [ 239],
       [  55],
       [ 101],
       [ 239],
       [ 607],
       [ 377],
       [ 929],
       [ 193],
       [ 285],
       [   9],
       [ 285],
       [ 193],
       [  55],
       [ 423],
       [   9],
       [ 929],
       [   9],
       [ 147],
       [  55],
       [ 193],
       [ 285],
       [ 653],
       [ 377],
       [   9],
       [ 239],
       [  55],
       [   9],
       [   9],
       [ 561],
       [ 101],
       [ 147],
       [ 699],
       [ 285],
       [   9],
       [ 377],
       [ 561],
       [   9],
       [ 193],
       [ 423],
       [ 193],
       [ 239],
       [1113],
       [ 101],
       [ 193],
       [   9],
       [ 285],
       [  55],
       [   9],
       [  55],
       [ 101],
       [ 147],
       [ 1